In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [ ]:
len(documents)

In [ ]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [ ]:
doc = documents[0]
doc
print(doc["content"])
print(doc["filename"])


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
import json

user_prompt = json.dumps(doc)

user_prompt

'{"content": "# Introduction\\n\\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\\n\\nIn this module, we\'ll build a working Retrieval-Augmented\\nGeneration (RAG) system from scratch, step by step.\\n\\nWe write everything in plain Python. We build a small search index by\\nhand and call the LLM ourselves. I want you to see every piece first.\\nThat way you know what a framework does for you before you reach for\\none.\\n\\nPlaces where you can find me:\\n\\n- [My substack](https://alexeyondata.substack.com/)\\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\\n- [X](https://x.com/Al_Grigor)\\n\\n## LLMs\\n\\nAn LLM (Large Language Model) is a neural network trained on massive\\namounts of text. Given a prompt, it generates a continuation - a\\nplausible next piece of text.\\n\\nThink of your phone. When you type \\"how are\\" in WhatsApp, it suggests\\n\\"you\\" as the next word. \\"How are you\\" is the most common

In [10]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [11]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [12]:
result = response.output_parsed

print(result)

questions=['What is a retrieval-augmented generation system, in simple terms?', 'Why does the course build the RAG project in plain Python instead of using a framework right away?', 'What are the main limits of large language models that RAG is meant to fix?', 'What kind of example project will this module build to show RAG in action?', 'What do the two parts of this module focus on, and how are they different?']


In [13]:
print(result.questions)

['What is a retrieval-augmented generation system, in simple terms?', 'Why does the course build the RAG project in plain Python instead of using a framework right away?', 'What are the main limits of large language models that RAG is meant to fix?', 'What kind of example project will this module build to show RAG in action?', 'What do the two parts of this module focus on, and how are they different?']


In [14]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['What is a RAG system, and how does it help with the limits of an LLM?', 'Why does the course treat the model like a black box instead of explaining how it works inside?', 'What kind of project are you building in this module to demonstrate RAG?', 'What will the first part of the module cover before moving on to the agentic version?', 'What changes in Part 2 when the pipeline becomes agentic?']


In [15]:
print(usage)

ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=101, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1121)


In [16]:
from evaluation_utils import calc_price

cost = calc_price(usage)

cost

{'input_cost': 0.0007650000000000001,
 'output_cost': 0.0004545,
 'total_cost': 0.0012195}

In [18]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["filename"]
    })

records

[{'question': 'What is a RAG system, and how does it help with the limits of an LLM?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does the course treat the model like a black box instead of explaining how it works inside?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What kind of project are you building in this module to demonstrate RAG?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will the first part of the module cover before moving on to the agentic version?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What changes in Part 2 when the pipeline becomes agentic?',
  'document': '01-agentic-rag/lessons/01-intro.md'}]

In [22]:
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [23]:
generate_ground_truth(doc)

([{'question': 'What is a Retrieval-Augmented Generation system, and how does it help with LLM limitations?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'Why do LLMs sometimes give wrong answers or miss information, and what are the main drawbacks mentioned here?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'Why does this course build the RAG system in plain Python instead of using a framework right away?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What is the FAQ agent in this module supposed to do for course students?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What topics are covered in the first part of the module, and what changes when it becomes more agent-like in part two?',
   'document': '01-agentic-rag/lessons/01-intro.md'}],
 ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=117, output_tokens_details=OutputTokensD

In [27]:
# Question 1
target_filenames = {
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
}

generated_records = []

for doc in documents:
    if doc["filename"] in target_filenames:
        ground_truth, usage = generate_ground_truth(doc)
        generated_records.extend(ground_truth)
        print(doc["filename"], usage)

generated_records

01-agentic-rag/lessons/01-intro.md ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=99, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1119)
01-agentic-rag/lessons/02-environment.md ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=130, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1416)
01-agentic-rag/lessons/03-rag.md ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=1280), output_tokens=107, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1860)


[{'question': 'What does RAG do to help an LLM answer questions more reliably?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why do LLMs sometimes get facts wrong or miss info from my documents?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does this course use the model through an API instead of training or hosting one locally?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main pieces you build in this module’s first part of the FAQ assistant?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What changes in the second part when the RAG pipeline becomes more agent-like?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What do I need installed before starting this module, and do I need anything besides Python and Jupyter?',
  'document': '01-agentic-rag/lessons/02-environment.md'},
 {'question': 'How do I set up a new project for the course from an empty fol

In [31]:
import pandas as pd

# Load CSV into a DataFrame
df = pd.read_csv("data/ground-truth.csv")

# Convert DataFrame rows into a list of dictionaries
ground_truth = df.to_dict(orient="records")

In [32]:
print(ground_truth[0])

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?", 'filename': '01-agentic-rag/lessons/01-intro.md'}


In [33]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295